In [2]:
import ee 
import geemap
ee.Authenticate()
ee.Initialize()

# Create map
Map = geemap.Map(center=[13.2, -59.5], zoom=10)

# Barbados boundary from GAUL
barbados = (
    ee.FeatureCollection('FAO/GAUL/2015/level0')
    .filter(ee.Filter.eq('ADM0_NAME', 'Barbados'))
)

# Convert Barbados feature collection to geometry
barbados_geom = barbados.geometry()

# Soil layers clipped to Barbados
# Organic carbon (dg/kg)
pa = (
    ee.Image("projects/deforestation-495419/assets/barbados_soil_org_C")
    .clip(barbados_geom)
)

# Nitrogen (cg/kg)
pn = (
    ee.Image("projects/deforestation-495419/assets/barbados_nitrogen")
    .clip(barbados_geom)
)

# Sand (g/kg)
ps = (
    ee.Image("projects/deforestation-495419/assets/barbados_sand_content")
    .clip(barbados_geom)
)

# pH (*10)
ph = (
    ee.Image("projects/deforestation-495419/assets/barbados_pH")
    .clip(barbados_geom)
)

# Visualization parameters
pa_vis = {
    "min": 0,
    "max": 150,
    "palette": [
        "#fbd1cc",
        "#dd9889",
        "#e2685d",
        "#bc3535",
        "#680000"
    ]
}

pn_vis = {
    "min": 0,
    "max": 80,
    "palette": [
        "#f7fcf5",
        "#c7e9c0",
        "#74c476",
        "#238b45",
        "#00441b"
    ]
}

ps_vis = {
    "min": 0,
    "max": 1000,
    "palette": [
        "#fff5eb",
        "#fdd0a2",
        "#fdae6b",
        "#e6550d",
        "#7f2704"
    ]
}

ph_vis = {
    "min": 40,
    "max": 80,
    "palette": [
        "#d73027",
        "#fc8d59",
        "#fee08b",
        "#d9ef8b",
        "#1a9850"
    ]
}

# Add layers
Map.addLayer(pa, pa_vis, "Soil Organic Carbon")
Map.addLayer(pn, pn_vis, "Nitrogen")
Map.addLayer(ps, ps_vis, "Sand")
Map.addLayer(ph, ph_vis, "pH")

# Add Barbados outline
Map.addLayer(
    barbados.style(**{
        "color": "black",
        "fillColor": "00000000",
        "width": 2
    }),
    {},
    "Barbados Boundary"
)

# Display map
Map

# Organic Carbon
task_pa = ee.batch.Export.image.toDrive(
    image=pa,
    description='Barbados_Soil_Organic_Carbon',
    folder='GEE_Exports',
    fileNamePrefix='soil_organic_carbon',
    region=barbados_geom,
    scale=30,      # adjust to your dataset resolution
    maxPixels=1e13
)
task_pa.start()

# Nitrogen
task_pn = ee.batch.Export.image.toDrive(
    image=pn,
    description='Barbados_Nitrogen',
    folder='GEE_Exports',
    fileNamePrefix='nitrogen',
    region=barbados_geom,
    scale=30,
    maxPixels=1e13
)
task_pn.start()

# Sand
task_ps = ee.batch.Export.image.toDrive(
    image=ps,
    description='Barbados_Sand',
    folder='GEE_Exports',
    fileNamePrefix='sand_content',
    region=barbados_geom,
    scale=30,
    maxPixels=1e13
)
task_ps.start()

# pH
task_ph = ee.batch.Export.image.toDrive(
    image=ph,
    description='Barbados_pH',
    folder='GEE_Exports',
    fileNamePrefix='soil_pH',
    region=barbados_geom,
    scale=30,
    maxPixels=1e13
)
task_ph.start()